<div align="center">

<img src="https://capsule-render.vercel.app/api?type=waving&height=280&color=gradient&text=𝗭-𝗜𝗺𝗮𝗴𝗲%20𝗣𝗿𝗼&fontAlignY=30&fontSize=100&desc=Next-Gen%20FP8%20·%20ComfyUI%20Backend%20·%20Smart%20Caching&descSize=30" />

<br/><br/>

[![Open in Colab](https://img.shields.io/badge/Google_Colab-F9AB00?style=for-the-badge&logo=googlecolab&logoColor=black)](https://colab.research.google.com/github/Shineii86/ZImagePro/blob/main/notebook/ZImagePro.ipynb)
![Model](https://img.shields.io/badge/Model-Z--Image%20Turbo%20Pro-A855F7?style=for-the-badge&logo=huggingface&logoColor=white)
![ComfyUI](https://img.shields.io/badge/Powered%20by-ComfyUI-FF6F00?style=for-the-badge)
![GPU](https://img.shields.io/badge/GPU-T4%20Required-76B900?style=for-the-badge&logo=nvidia&logoColor=white)
![Python](https://img.shields.io/badge/Python-3.10%2B-3776AB?style=for-the-badge&logo=python&logoColor=white)
![License](https://img.shields.io/badge/License-MIT-blue?style=for-the-badge&logo=gnu&logoColor=white)

</div>

## 🚀 What is Z-Image Turbo Pro?

Next-gen FP8 diffusion pipeline with ComfyUI backend and smart caching. Professional-grade image generation on free Colab.

> **Why?** FP8 quantization cuts VRAM usage nearly in half while preserving output quality — enabling pro-grade generation on free hardware.

**Key Features:**
- 🔋 **GPU Ready** — Runs on free T4 Colab
- ⚡ **FP8 Optimized** — Half the VRAM
- 💾 **Smart Cache** — Models cached after first run
- 🎯 **One-Click** — Zero configuration

---

### 📐 Supported Resolutions

| Ratio | Resolution | Best For |
|:-----:|:----------:|----------|
| 1:1 | 1024×1024 | Avatars, social posts |
| 16:9 | 1280×720 | Thumbnails, wallpapers |
| 9:16 | 720×1280 | Mobile, stories |
| 4:3 | 1152×864 | Classic photo |
| 21:9 | 1344×576 | Ultrawide, cinematic |

---

### ⚙️ How It Works

```mermaid
flowchart LR
    A[📝 Input] --> B[Process]
    B --> C[Generate]
    C --> D[💾 Output]
```

In [ ]:
#@title 🛠️ 1. Initialize
#@markdown <span style='color:#94a3b8;font-size:13px;'>Set up core environment & download models</span>

from IPython.display import display, HTML
display(HTML('''
<div style="background:#064e3b; color:#6ee7b7; padding:12px 16px; border-radius:8px; font-family:monospace; font-size:14px; margin-bottom:8px;">
  🛠️ <b>Step 1/3</b> — Initialize<br>
  <span style="color:#94a3b8; font-size:12px;">Set up core environment & download models</span>
</div>
'''))

import os, sys

# Clone repo (for Colab) and add src/ to path
REPO_DIR = "/content/ZImagePro"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Shineii86/ZImagePro.git {REPO_DIR} &> /dev/null
    print("   ✓ Repository Cloned")
else:
    !cd {REPO_DIR} && git pull &> /dev/null
    print("   ✓ Repository Updated")

sys.path.insert(0, REPO_DIR)

LOCAL_WORKSPACE = "/content/ComfyUI"

print("🚀 Initializing Core Architecture...")
if not os.path.exists(LOCAL_WORKSPACE):
    !git clone https://github.com/comfyanonymous/ComfyUI {LOCAL_WORKSPACE} &> /dev/null
    print("   ✓ Core Engine Cloned")
else:
    !cd {LOCAL_WORKSPACE} && git pull &> /dev/null
    print("   ✓ Core Engine Updated")

print("📦 Installing Dependencies (This takes a moment)...")
!cd {LOCAL_WORKSPACE} && pip install xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121 &> /dev/null

from src.config import MODEL_DIRS, UNET_URL, TEXT_ENCODER_URL, VAE_URL
from src.downloader import ensure_aria2, download_file

ensure_aria2()

# Download all models
dirs = {k: os.path.join(LOCAL_WORKSPACE, v) for k, v in MODEL_DIRS.items()}

print("📥 Fetching Models...")
download_file(UNET_URL,         dirs["unet"])
download_file(TEXT_ENCODER_URL,  dirs["clip"])
download_file(VAE_URL,           dirs["vae"])

print("✅ Environment Ready! Please load the engine below.")

> ⚠️ **Content Safety Notice**
>
> Z-Image is an **unfiltered diffusion model** — it does not have built-in NSFW filters.
> You are **solely responsible** for the content you generate.
>
> **Do NOT** use this tool to create:
> - Illegal content
> - Harmful, abusive, or violent content
> - Non-consensual intimate imagery
> - Content involving minors in any form
>
> By running this notebook, you agree to comply with all applicable laws
> and the [HuggingFace Content Guidelines](https://huggingface.co/content-guidelines).
> The authors assume no liability for misuse.

In [ ]:
#@title 🚀 2. Load Engine & Generate
#@markdown <span style='color:#94a3b8;font-size:13px;'>Load FP8 weights, configure prompt, and generate</span>

from IPython.display import display, HTML
display(HTML('''
<div style="background:#1e3a5f; color:#93c5fd; padding:12px 16px; border-radius:8px; font-family:monospace; font-size:14px; margin-bottom:8px;">
  🚀 <b>Step 2/3</b> — Load Engine & Generate<br>
  <span style="color:#94a3b8; font-size:12px;">Load FP8 weights, configure prompt, and generate</span>
</div>
'''))

import sys, os
sys.path.insert(0, "/content/ZImagePro")

from src.config import WORKSPACE
from src.generator import load_models, generate_image

os.chdir(WORKSPACE)

# Load models
nodes, unet_model, clip_model, vae_model = load_models()

# --- Generation Settings ---
positive_prompt = "A cinematic shot of a futuristic neon city, rain reflections, cybernetic aesthetics, 8k masterpiece" # @param {type:"string"}
negative_prompt = "blurry, low quality, text, watermark, distorted" # @param {type:"string"}

#@markdown ### 📐 **Dimensions & Quality**
aspect_ratio = "1280x720 (16:9 Landscape)" # @param ["1024x1024 (1:1 Square)", "1280x720 (16:9 Landscape)", "720x1280 (9:16 Portrait)", "1152x864 (4:3 Photo)", "1344x576 (21:9 Cinema)"]
steps = 20 # @param {type:"slider", min:10, max:50, step:1}
guidance_scale = 1 # @param {type:"slider", min:1.0, max:10.0, step:0.5}

#@markdown ### ⚙️ **Advanced**
seed = -1 # @param {type:"number"}
auto_download = False # @param {type:"boolean"}

# Parse resolution
w, h = [int(x) for x in aspect_ratio.split("(")[0].strip().split("x")]

img, path = generate_image(
    nodes=nodes,
    unet_model=unet_model,
    clip_model=clip_model,
    vae_model=vae_model,
    prompt=positive_prompt,
    negative_prompt=negative_prompt,
    width=w,
    height=h,
    steps=steps,
    cfg=guidance_scale,
    seed=seed,
)

display(img)

if auto_download:
    from google.colab import files
    files.download(path)

In [ ]:
#@title 💾 3. Export
#@markdown <span style='color:#94a3b8;font-size:13px;'>Download all results</span>

from IPython.display import display, HTML
display(HTML('''
<div style="background:#3b0764; color:#d8b4fe; padding:12px 16px; border-radius:8px; font-family:monospace; font-size:14px; margin-bottom:8px;">
  💾 <b>Step 3/3</b> — Export<br>
  <span style="color:#94a3b8; font-size:12px;">Download all results</span>
</div>
'''))

from src.exporter import zip_outputs, download_zip

zip_path = zip_outputs()
if zip_path:
    download_zip(zip_path)

---

<div align="center">

[![Open in Colab](https://img.shields.io/badge/Google_Colab-F9AB00?style=for-the-badge&logo=googlecolab&logoColor=black)](https://colab.research.google.com/github/Shineii86/ZImagePro/blob/main/notebook/ZImagePro.ipynb)
[![Back to ZImagePro](https://img.shields.io/badge/Back_to_ZImagePro-181717?style=for-the-badge&logo=github&logoColor=white)](../../README.md)
[![Documentation](https://img.shields.io/badge/README-238636?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Shineii86/ZImagePro)
[![Issues](https://img.shields.io/badge/Issues-DA3633?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Shineii86/ZImagePro/issues)
[![Profile](https://img.shields.io/badge/Shinei_Nouzen-8957E5?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Shineii86)

<br/><br/>

> All generated outputs are saved in `/content/results`. Download via the file explorer sidebar.

*Licensed under MIT · Made with ⚡ by [Shinei Nouzen](https://github.com/Shineii86)*

</div>